In [1]:
import os
import pdfplumber
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import words
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
def extract_text_from_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        text = ''
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text
    return text

def read_pdfs_in_directory(directory):
    pdf_texts = []
    pdf_filenames = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.pdf'):
                file_path = os.path.join(root, file)
                print(f"Reading {file_path}")
                text = extract_text_from_pdf(file_path)
                pdf_texts.append(text)
                pdf_filenames.append(file) 
    return pdf_texts, pdf_filenames


directory1 = "C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC"  # Replace with your actual PDF folder path

print("Processing the first folder...")
pdf_texts_1, pdf_files_1 = read_pdfs_in_directory(directory1)

Processing the first folder...
Reading C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Abarca-Cabrera-2023-Biocorona on Iron Oxide Na.pdf
Reading C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Abbina-2020-Blood circulation of soft nanomate.pdf
Reading C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Abdelkhaliq-2018-Impact of nanoparticle surfac.pdf
Reading C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Abedi et al. - 2019 - Citric acid functionalized silane coupling versus post-grafting strategy for dual pH and saline resp.pdf
Reading C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Abraham-2018-Phytochemicals as Dynamic Surface.pdf
Reading C:/Users/huang/OneDrive/Desktop/HIM/Proj

[WARNING] Metadata key "ModDate" could not be parsed due to exception: maximum recursion depth exceeded


Reading C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Aliyandi et al. - 2023 - Cell surface biotinylation to identify the receptors involved in nanoparticle uptake into endothelia.pdf
Reading C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Aliyandi-2021-Correlating Corona Composition a.pdf
Reading C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Allouni-2015-The effect of blood protein adsor.pdf
Reading C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Almalik-2017-Hyaluronic Acid Coated Chitosan N.pdf
Reading C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Alwani et al. - 2019 - Lysine functionalized nanodiamonds as gene carriers - Investigation of internalization pathways and.pdf


In [3]:
def clean_text_with_nltk(text):
    # Replace newline characters with a space
    text = text.replace('\n', ' ')
    # Remove URLs, emails, and other similar patterns
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S*@\S*\s?', '', text)  # Remove email addresses
    # Handle specific problematic phrases or common artifacts
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove non-alphabetic characters but keep spaces
    text = text.lower().strip()
    
    # Tokenize the text
    tokens = word_tokenize(text)
    # Remove very short tokens
    clean_tokens = [t for t in tokens if len(t) > 4]
    
    # Rejoin tokens into a clean string
    processed_text = ' '.join(clean_tokens)
    return processed_text

def keras_tokenize(texts):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    word_index = tokenizer.word_index
    
    # Convert sequences back to words, filtering out words that weren't tokenized
    processed_texts = []
    for seq in sequences:
        words_list = [tokenizer.index_word.get(i, '') for i in seq if i in tokenizer.index_word]
        processed_texts.append(' '.join(words_list))
    return processed_texts

pdf_texts_1_clean = [clean_text_with_nltk(text) for text in pdf_texts_1]
pdf_texts_1_tokenized = keras_tokenize(pdf_texts_1_clean)


labels = [0] * len(pdf_texts_1_clean)  
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X = vectorizer.fit_transform(pdf_texts_1_tokenized)
X1 = vectorizer.transform(pdf_texts_1_tokenized)


In [4]:
df=pd.DataFrame(pdf_texts_1_clean, columns=['text'])
df.to_csv('pdf_texts_cleaned.csv', index=False)
# 你要检查的关键词
keywords = ['lcmsms', 'proteomic', 'proteomics']

# 用一个空列表来存储符合条件的索引
matching_indices = []

# 循环遍历 DataFrame 中的每一行
for index, row in df.iterrows():
    text = row['text'].lower()  # 转换为小写，方便匹配
    if any(keyword in text for keyword in keywords):  # 检查是否有任何一个关键词出现
        matching_indices.append(index)
    
df_prot=df.iloc[matching_indices,:]

len(df_prot)

df_notprot=df.drop(matching_indices)

len(df_notprot)

df_prot.to_csv('proteomics.csv',header=False)

df_notprot.to_csv('notproteomics.csv',header=False)

In [7]:
import shutil

target_directory = "C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Proteomics related"

# 如果目标文件夹不存在，创建目标文件夹
if not os.path.exists(target_directory):
    os.makedirs(target_directory)


def move_files_to_new_folder(pdf_filenames, matching_indices, source_directory, target_directory):
    # 确保目标目录存在
    if not os.path.exists(target_directory):
        os.makedirs(target_directory)
    
    # 遍历所有匹配的索引
    for idx in matching_indices:
        # 获取对应的文件名
        pdf_filename = pdf_filenames[idx]
        
        # 遍历source_directory中的所有子文件夹
        for root, dirs, files in os.walk(source_directory):
            if pdf_filename in files:
                # 构造源文件路径
                source_file = os.path.join(root, pdf_filename)
                
                # 构造目标文件路径
                target_file = os.path.join(target_directory, pdf_filename)
                
                # 移动文件
                shutil.move(source_file, target_file)
                print(f"Moved {source_file} to {target_file}")
                break  # 找到文件后跳出子文件夹的循环

# 运行文件移动函数
move_files_to_new_folder(pdf_files_1, matching_indices, directory1, target_directory)

Moved C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Dell'Orco et al. - 2012 - Delivery success rate of engineered nanoparticles in the presence of the protein corona a systems-l.pdf to C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Proteomics related\Dell'Orco et al. - 2012 - Delivery success rate of engineered nanoparticles in the presence of the protein corona a systems-l.pdf
Moved C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\Dell'Orco-2010-Modeling the Time Evolution of.pdf to C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Proteomics related\Dell'Orco-2010-Modeling the Time Evolution of.pdf
Moved C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/Data analysis/Final_PDF_NP-PC\DeLoid-2022-Incineration-Generated Polyethylen.pdf to C:/Use

In [4]:
def extract_top_features_tfidf(vectorizer, X, n_features=20):
    feature_names = vectorizer.get_feature_names_out()
    max_tfidf = X.max(0).toarray().ravel()  # Convert to dense array and find the max tf-idf score across all docs
    sorted_by_tfidf = max_tfidf.argsort()   # Sort features by their max tf-idf
    top_features = [(feature_names[i], max_tfidf[i]) for i in sorted_by_tfidf[-n_features:]]
    return top_features

top_features_tfidf1 = extract_top_features_tfidf(vectorizer, X1, n_features=2000)

def plot_top_features(features, title, color):
    features = sorted(features, key=lambda x: x[1], reverse=True)
    words, scores = zip(*features)
    plt.figure(figsize=(10, 40))
    bars = plt.barh(words, scores, color=color, edgecolor='black')
    plt.xlabel('Score')
    plt.title(title)
    plt.gca().invert_yaxis()

    for bar in bars:
        plt.text(
            bar.get_width() + 0.01, 
            bar.get_y() + bar.get_height()/2, 
            f'{bar.get_width():.4f}', 
            va='center'
        )
    plt.show()


In [5]:
top = pd.read_csv("C:/Users/huang/OneDrive/Desktop/HIM/Project/NP-PC data mining/formal experiment/biological identity_top_features.csv")
top_features_only = top['Feature']

#  6.2) 使用 CountVectorizer 只统计这些词的出现次数
count_vectorizer = CountVectorizer(vocabulary=top_features_only, stop_words='english')
X_counts = count_vectorizer.fit_transform(pdf_texts_1_tokenized)  # (文档数 x top_features数)

#  6.3) 将计数矩阵转成DataFrame
df_counts = pd.DataFrame(X_counts.toarray(), columns=top_features_only)

#  6.4) 在第一列插入 PDF 文件名
df_counts.insert(0, 'PDF_Name', pdf_files_1)

#  6.5) 导出到 CSV
output_csv_path = "top_features_counts_per_pdf_year.csv"
df_counts.to_csv(output_csv_path, index=False, encoding='utf-8')
print(f"Done! CSV saved to: {output_csv_path}")

Done! CSV saved to: top_features_counts_per_pdf_year.csv
